# 04 · Discriminator
Residual blocks, filtered downsampling, minibatch stddev, scalar logit.

In [ ]:
import os, sys
# ---- platform auto-detect: the same notebook runs on Colab and Kaggle ----
PLATFORM = "kaggle" if os.path.exists("/kaggle/input") else "colab"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if (PLATFORM == "colab" or PLATFORM == "kaggle") and not os.path.exists("src"):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Ravikishore710/styleforge3-T.git"], check=True)
    os.chdir("styleforge3-T")
sys.path.insert(0, os.path.abspath("."))
!pip install -q -r requirements.txt
import tensorflow as tf
print("platform:", PLATFORM, "| TF:", tf.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU"))
# Kaggle: enable GPU (Settings -> Accelerator -> GPU P100) and add the FFHQ
# dataset to /kaggle/input, or run scripts/prepare_ffhq.py --source folder.

In [ ]:
from src.config import load_config
from src.discriminator.discriminator import Discriminator
import tensorflow as tf
cfg = load_config('configs/ffhq_64.yaml')
D = Discriminator(cfg)
x = tf.random.normal([4, 64, 64, 3])
print('logits:', D(x).shape, '| D params:', f'{D.count_params():,}')

In [ ]:
import numpy as np
one = tf.random.normal([1, 64, 64, 3])
solo = D(one).numpy()[0, 0]
in_batch = D(tf.tile(one, [4, 1, 1, 1])).numpy()[0, 0]
print('solo:', solo, '| in identical batch:', in_batch)
assert np.isclose(solo, in_batch)

In [ ]:
with tf.GradientTape() as tape:
    loss = tf.reduce_mean(D(x, training=True))
grads = tape.gradient(loss, D.trainable_variables)
print('grads present:', all(g is not None for g in grads),
      '| finite:', all(tf.reduce_all(tf.math.is_finite(g)).numpy() for g in grads))